# DeepCare 04: Keras MLP

Trains the deep learning model and compares it with the baselines from `03_baselines.ipynb` under the **same protocol**:

- Hyperparameters chosen by random search with **Keras Tuner**, objective = **PR-AUC on the validation set**
- Class-weighted loss (no resampling), early stopping and learning-rate reduction on validation PR-AUC
- Decision threshold chosen on the validation set (maximizing F1), then applied unchanged to the test set
- Test set used once, for the final numbers
- Training logged to **TensorBoard**

Because neural network training is stochastic, the best configuration is retrained with three seeds. The main result uses seed 42, and the other two seeds show how much the result moves from run to run.

Note: Keras computes PR-AUC with a binned approximation, so the tuning objective differs slightly from the exact PR-AUC (scikit-learn's average precision) used in the results tables.

## 1. Setup and data

In [ ]:
!pip -q install keras-tuner

In [ ]:
import os, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras_tuner as kt
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_recall_curve,
                             confusion_matrix, f1_score)
from google.colab import drive
drive.mount('/content/drive')

SEED = 42
N_TRIALS = 10        # tuner trials (lower this if runtime is too long)
MAX_EPOCHS = 60
BATCH = 1024
PROJECT_DIR = '/content/drive/MyDrive/deepcare'
TARGET = 'Diabetes_binary'
D, R = f'{PROJECT_DIR}/data', f'{PROJECT_DIR}/results'
TUNER_DIR, TB_DIR = f'{PROJECT_DIR}/tuner', f'{PROJECT_DIR}/logs'
for p in [R, TUNER_DIR, TB_DIR, f'{PROJECT_DIR}/models']:
    os.makedirs(p, exist_ok=True)

tf.keras.utils.set_random_seed(SEED)

def load(split):
    df = pd.read_csv(f'{D}/{split}_scaled.csv')
    return df.drop(columns=[TARGET]).to_numpy(dtype='float32'), df[TARGET].to_numpy(dtype='float32')

Xtr, ytr = load('train'); Xva, yva = load('val'); Xte, yte = load('test')
n_features = Xtr.shape[1]

# balanced class weights (mean sample weight stays near 1)
n, n0, n1 = len(ytr), (ytr == 0).sum(), (ytr == 1).sum()
CW = {0: float(n / (2 * n0)), 1: float(n / (2 * n1))}

print('features:', n_features, '| train/val/test:', len(ytr), len(yva), len(yte))
print('class weights:', {k: round(v, 3) for k, v in CW.items()})
print('TensorFlow', tf.__version__)

## 2. Evaluation helpers (same as notebook 03)

In [ ]:
def best_threshold(y, p):
    prec, rec, thr = precision_recall_curve(y, p)
    f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
    return float(thr[np.argmax(f1)])

def metrics_at(y, p, t):
    pred = (p >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {'threshold': round(float(t), 4),
            'recall': tp / (tp + fn),
            'precision': tp / (tp + fp) if (tp + fp) else 0.0,
            'specificity': tn / (tn + fp),
            'f1': f1_score(y, pred),
            'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)}

def result_row(name, p_val, p_test):
    t_star = best_threshold(yva, p_val)
    row = {'model': name,
           'val_pr_auc': average_precision_score(yva, p_val),
           'roc_auc': roc_auc_score(yte, p_test),
           'pr_auc': average_precision_score(yte, p_test)}
    for tag, t in [('@0.5', 0.5), ('@tuned', t_star)]:
        for k, v in metrics_at(yte, p_test, t).items():
            row[f'{k}{tag}'] = v
    return row

## 3. Model and hyperparameter search space

Searched: number of hidden layers (1 to 3), units per layer (64/128/256), dropout, batch normalization on/off, L2 strength, and learning rate. Batch size is fixed at 1024.

In [ ]:
def build_model(hp):
    n_layers = hp.Int('n_layers', 1, 3)
    use_bn = hp.Boolean('batch_norm')
    dropout = hp.Choice('dropout', [0.1, 0.2, 0.3, 0.4, 0.5])
    l2 = hp.Choice('l2', [0.0, 1e-5, 1e-4])
    reg = tf.keras.regularizers.l2(l2) if l2 > 0 else None

    inputs = tf.keras.Input(shape=(n_features,))
    x = inputs
    for i in range(n_layers):
        x = tf.keras.layers.Dense(hp.Choice(f'units_{i}', [64, 128, 256]), kernel_regularizer=reg)(x)
        if use_bn:
            x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.Dropout(dropout)(x)
    outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(hp.Float('lr', 1e-4, 3e-3, sampling='log')),
                  loss='binary_crossentropy',
                  metrics=[tf.keras.metrics.AUC(name='roc_auc'),
                           tf.keras.metrics.AUC(curve='PR', name='pr_auc')])
    return model

def make_callbacks():
    return [tf.keras.callbacks.EarlyStopping(monitor='val_pr_auc', mode='max', patience=8,
                                             restore_best_weights=True),
            tf.keras.callbacks.ReduceLROnPlateau(monitor='val_pr_auc', mode='max', factor=0.5,
                                                 patience=3, min_lr=1e-6)]

build_model(kt.HyperParameters()).summary()

## 4. Hyperparameter search

Each trial logs to TensorBoard under its own folder. Results persist in Drive: if Colab disconnects, rerun this cell and the search resumes from the finished trials. To start the search over, delete the `tuner/` folder in Drive first.

In [ ]:
tuner = kt.RandomSearch(build_model,
                        objective=kt.Objective('val_pr_auc', 'max'),
                        max_trials=N_TRIALS, seed=SEED,
                        directory=TUNER_DIR, project_name='mlp', overwrite=False)

tuner.search(Xtr, ytr, validation_data=(Xva, yva), epochs=MAX_EPOCHS, batch_size=BATCH,
             class_weight=CW, verbose=0,
             callbacks=make_callbacks() + [tf.keras.callbacks.TensorBoard(log_dir=f'{TB_DIR}/tuning')])

tuner.results_summary(5)
best_hp = tuner.get_best_hyperparameters(1)[0]
print('\nbest hyperparameters:', best_hp.values)

## 5. Retrain the best configuration with three seeds

Seed 42 is the main model. The other seeds measure run-to-run variation, which tells you how large a gap to XGBoost has to be before it means anything.

In [ ]:
SEEDS = [42, 7, 123]
runs = {}
for s in SEEDS:
    tf.keras.utils.set_random_seed(s)
    m = build_model(best_hp)
    h = m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=MAX_EPOCHS, batch_size=BATCH,
              class_weight=CW, verbose=0,
              callbacks=make_callbacks() + [tf.keras.callbacks.TensorBoard(log_dir=f'{TB_DIR}/final_seed{s}')])
    p_val = m.predict(Xva, batch_size=4096, verbose=0).ravel()
    p_te = m.predict(Xte, batch_size=4096, verbose=0).ravel()
    runs[s] = {'model': m, 'hist': h.history, 'p_val': p_val, 'p_te': p_te}
    print(f'seed {s:3d}: epochs={len(h.history["loss"]):2d}  '
          f'test ROC-AUC={roc_auc_score(yte, p_te):.4f}  test PR-AUC={average_precision_score(yte, p_te):.4f}')

roc = np.array([roc_auc_score(yte, r['p_te']) for r in runs.values()])
pr = np.array([average_precision_score(yte, r['p_te']) for r in runs.values()])
print(f'\nacross seeds: ROC-AUC {roc.mean():.4f} +/- {roc.std():.4f} | PR-AUC {pr.mean():.4f} +/- {pr.std():.4f}')

## 6. Main model: evaluation, curves and saving

In [ ]:
main = runs[42]
row = result_row('mlp', main['p_val'], main['p_te'])
np.savez(f'{R}/probs_mlp.npz', val=main['p_val'], test=main['p_te'], val_y=yva, test_y=yte)
main['model'].save(f'{PROJECT_DIR}/models/mlp.keras')

with open(f'{R}/mlp_params.json', 'w') as f:
    json.dump({'hyperparameters': best_hp.values, 'class_weights': CW, 'batch_size': BATCH,
               'epochs_trained_seed42': len(main['hist']['loss']),
               'seed_variation': {'test_roc_auc': roc.tolist(), 'test_pr_auc': pr.tolist(), 'seeds': SEEDS}},
              f, indent=2, default=str)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for key, label in [('loss', 'train'), ('val_loss', 'validation')]:
    ax[0].plot(main['hist'][key], label=label)
ax[0].set_title('Loss'); ax[0].set_xlabel('epoch'); ax[0].legend()
for key, label in [('pr_auc', 'train'), ('val_pr_auc', 'validation')]:
    ax[1].plot(main['hist'][key], label=label)
ax[1].set_title('PR-AUC (Keras)'); ax[1].set_xlabel('epoch'); ax[1].legend()
plt.tight_layout(); plt.savefig(f'{R}/mlp_training_curves.png', dpi=150); plt.show()

## 7. Comparison with the baselines

In [ ]:
base = pd.read_csv(f'{R}/baseline_results.csv', index_col='model')
allres = pd.concat([base, pd.DataFrame([row]).set_index('model')])
allres.to_csv(f'{R}/all_results.csv')

cols = ['roc_auc', 'pr_auc', 'recall@0.5', 'precision@0.5', 'threshold@tuned',
        'recall@tuned', 'precision@tuned', 'specificity@tuned', 'f1@tuned']
print('no-skill PR-AUC (= test prevalence):', round(float(yte.mean()), 4))
display(allres[cols].sort_values('pr_auc', ascending=False).round(4))
print('saved to', R)

## 8. TensorBoard (optional)

Opens the logged tuning trials and final runs. In the HParams tab you can compare trials side by side.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/deepcare/logs